In [0]:
%sql
DROP TABLE IF EXISTS la_lakehouse.silver.silver_permits_issued_enriched;

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

In [0]:
parcel_df = spark.table("la_lakehouse.silver.la_parcels")
la_permit_issued_df = spark.table("la_lakehouse.silver.la_building_permits_issued") 

In [0]:
la_permit_issued_df = la_permit_issued_df.withColumnRenamed("_updated_at", "permit_updated_at")
parcel_df = parcel_df.withColumnRenamed("_updated_at", "parcel_updated_at")

In [0]:
silver_join_df = la_permit_issued_df.join(parcel_df, la_permit_issued_df.pin_nbr == parcel_df.pin, "left")
display(silver_join_df)

#Testing Before Silver Table

In [0]:
permit_count = la_permit_issued_df.count()
joined_count = silver_join_df.count()
print(f"Permits: {permit_count}, Joined: {joined_count}, Match rate: {joined_count/permit_count*100:.2f}%")

In [0]:
silver_join_df.filter(col("permit_nbr").isNull()).count()

In [0]:
parcel_df.groupBy("pin").count().filter("count > 1").count()

In [0]:
la_permit_issued_df.groupBy("pin_nbr").count().filter("count > 1").count()

In [0]:
parcel_df.count()

In [0]:
silver_join_df.groupBy("permit_nbr").count().filter("count > 1").show(20)

In [0]:
print(la_permit_issued_df.count())
la_permit_issued_df.select("permit_nbr", "pin_nbr").show(5)

In [0]:
matched = silver_join_df.filter(col("pin").isNotNull()).count()
print(f"Rows with a real parcel match: {matched} / {405688} = {matched/405688*100:.2f}%")

In [0]:
silver_join_df.filter(col("pin").isNull()).groupBy("pin_nbr").count().orderBy(F.desc("count")).show(10)

#Rename update_at

#Writing to the Silver Table

In [0]:
(
    silver_join_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("la_lakehouse.silver.silver_permits_issued_enriched")
)

#Quick-Looking

In [0]:
spark.table("la_lakehouse.silver.silver_permits_issued_enriched").count()

In [0]:
persisted_df = spark.table("la_lakehouse.silver.silver_permits_issued_enriched")
matched = persisted_df.filter(col("pin").isNotNull()).count()
total = persisted_df.count()
print(f"Rows with a real parcel match: {matched} / {total} = {matched/total*100:.2f}%")

In [0]:
spark.table("la_lakehouse.silver.silver_permits_issued_enriched").printSchema()